# Fig 2F — Finite-reference uncertainty benchmark (data processing)

UK Biobank simulation with a B = 500 sketch reference: power / FDR of SuSiE-rRSS against all QC methods across NCP. Bootstrap 95% CIs over replicates.

**Output:** `figure_data/Fig2C.Finite_Reference_Benchmark.rds` (+ `.csv`)

In [ ]:
source("../../R/utils.R")

data_name <- "Fig2C.Finite_Reference_Benchmark"

carma         <- readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKBB_B500_CARMA/summary/result_single_QC_summary.rds")              %>% mutate(qc = "CARMA")
dentist_qc    <- readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKBB_B500_DEN_SLALOM//summary/result_DEN_SLA_summary_DENTIST_QC.rds")    %>% mutate(qc = "DEN_QC")
dentist_raiss <- readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKBB_B500_DEN_SLALOM//summary/result_DEN_SLA_summary_DENTIST_RAISS.rds") %>% mutate(qc = "DEN_RAISS")
slalom_raiss  <- readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKBB_B500_DEN_SLALOM//summary/result_DEN_SLA_summary_SLALOM_RAISS.rds")  %>% mutate(qc = "SLALOM_RAISS")
rsparse_pro   <- readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKBB_B500_rsparse_pro///summary/result_single_QC_summary.rds")           %>% mutate(qc = "Rsparse_Pro")
susie         <- readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKBB_B_finemap/summary/result_susie_cs_summary_summary.rds")

In [2]:
extract_h2g <- function(x) as.numeric(str_extract(x, "(?<=h2g_)[0-9.]+"))

# ---- QC methods: keep only the 'sketch' reference ----
others_std <- bind_rows(carma, dentist_qc, dentist_raiss, slalom_raiss, rsparse_pro) %>%
  filter(method == "sketch") %>%
  transmute(input_file, qc,
            h2g     = extract_h2g(input_file),
            ncausal,
            true_var_number, true_set_number,
            total_var_number, total_set_number, total_set_length)

# ---- susie: sketch -> SuSiE-RSS, sketch_mix -> SuSiE-rRSS ----
susie_std <- susie %>%
  filter(method %in% c("sketch", "sketch_mix"),
         B_stoc == 500) %>%                        # match the other methods' B
  transmute(input_file,
            qc      = recode(method, sketch = "SuSiE-RSS", sketch_mix = "SuSiE-rRSS"),
            h2g     = extract_h2g(input_file),
            ncausal = as.integer(str_extract(input_file, "(?<=causal_)\\d+")),
            true_var_number, true_set_number,
            total_var_number, total_set_number, total_set_length)

# ---- combine ----
cell_reps <- bind_rows(others_std, susie_std) %>%
  mutate(causal_k = total_var_number)              # alias survives group_modify

In [3]:
# ---- bootstrap (boot_cell() in R/utils.R) ----
n_boot <- 1000
set.seed(1)

summary_ci <- cell_reps %>%
  group_by(qc, h2g, total_var_number) %>%          # dropped 'method' (reference); h2g is the new dim
  group_modify(~ boot_cell(.x, B = n_boot)) %>%
  ungroup() %>%
  mutate(qc = factor(qc, levels = c("CARMA","DEN_QC","DEN_RAISS","SLALOM_RAISS",
                                    "Rsparse_Pro","SuSiE-RSS","SuSiE-rRSS")))

In [4]:
## ---- plot-ready labels / facets ---------------------------------------------
n_gwas <- 300000                                # UKB GWAS N; NCP = N * h2g

plot_df <- summary_ci %>%
  mutate(qc = recode(as.character(qc),
           DEN_RAISS    = "DENTIST + RAISS",
           SLALOM_RAISS = "SLALOM + RAISS",
           Rsparse_Pro  = "RSparsePro")) %>%
  filter(qc %in% names(pal_qc)) %>%
  mutate(qc      = factor(qc, levels = names(pal_qc)),
         h2g_num = as.numeric(h2g),
         ncp_num = n_gwas * h2g_num,
         ncp     = fct_reorder(paste0("NCP = ", formatC(ncp_num, format = "f", digits = 1)), ncp_num))
stopifnot(!any(is.na(plot_df$qc)))

save_data(plot_df, data_name)